# React — Redux Toolkit

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> **Where this topic happens.** Not in the shared playground — Redux is a dependency, and the
> playground stays React and Vite only. The store you build here is **Mini-project 4's** store,
> so the "in your project" blocks are the real thing rather than practice.
>
> The runnable cells are the parts of Redux that are plain JavaScript and worth writing by hand:
> reducers, the shape of the state, and selectors. Redux Toolkit's own API is shown as **The
> React API** blocks — importing npm packages is not something a notebook cell can do honestly.
>
> Verified while writing: **`@reduxjs/toolkit` 2.12.0** and **`react-redux` 9.3.0**, with React
> 19.3.0. Every behaviour quoted below was executed against those versions.

## LESSON 87 — When a store beats Context

You already have two ways to share state: lift it up (LESSON 29) and Context (LESSON 56). Both
work, and for most applications they are enough. This lesson is about the point where they stop
being enough, and about what a store adds.

### What Context actually gives you

Context is a **delivery mechanism**. It removes prop drilling; it does not manage anything. The
state still lives in a component, the update logic still lives in that component, and — LESSON 66
— every consumer re-renders when the value changes.

That is fine until you hit one of these:

| symptom | why Context struggles |
|---|---|
| many unrelated pieces of shared state | one provider per concern, and the tree fills with providers |
| updates from many places | the update functions live in one component and get passed everywhere |
| a consumer needs *one field* of a large value | it re-renders when any other field changes |
| you want to see what changed and when | there is nothing to look at — state changes are just `setState` calls |

### What a store adds

A store is one object holding the application state, changed only by dispatching **actions**
through **reducers**, with components **selecting** the parts they need.

The four properties that matter in practice:

1. **One place.** All the shared state, in one shape, independent of the component tree.
2. **One way to change it.** Every change is an action, so every change is describable — and
   loggable.
3. **Selective reading.** A component subscribes to the slice it selected, not to the whole store.
4. **Tooling.** The Redux DevTools show every dispatched action and the state before and after.
   That is the reason experienced teams reach for it more often than you might expect.

And what it costs: a dependency, a vocabulary, and a layer of indirection between a click and a
change. That trade is worth making for the symptoms above, and not worth making for a theme
toggle.

### The honest recommendation

> Start with `useState`. Lift it up when two components need it. Use Context when passing it down
> is the problem. Reach for a store when the *state itself* is the problem.

Mini-project 4 uses Redux Toolkit because it is a capstone and you should have written one — not
because a project of that size cannot be built with Context. Knowing where the line is matters
more than which side of it you land on.

### Getting the state's shape right first

Before any API: a store is only as good as the shape of the state in it, and the shape that causes
trouble is nested. LESSON 66 quoted React's rule — *avoid deeply nested state* — and in a store it
matters more, because everything reads from the same object.

The fix is **normalisation**: store things by id in a lookup object, and reference them by id
everywhere else, the way a database does. Deleting a project then touches one place instead of
four, and two components looking at the same task cannot disagree.

The example cell does this conversion and counts what it saves. It is plain JavaScript, and it is
the part of Redux that is genuinely worth practising before you install anything.

### Key Notes

- Context delivers state; a store manages it. They solve different problems.
- Reach for a store for: many pieces of shared state, updates from everywhere, selective reads,
  and traceable changes.
- The cost is a dependency and a layer of indirection. A theme toggle does not need it.
- Normalise the state — by id, flat — before you write a single reducer.

### Example

**Runnable — plain JS.** Nested state, normalised state, and the same edit performed against
both. No Redux here: this is the design work that has to happen before the library is any use.

In [ ]:
// L87 — normalising the state a store will hold

const l87Nested = {
  projects: [
    {
      id: "p1",
      name: "Apollo",
      tasks: [
        { id: "t1", title: "Design", assignee: { id: "u1", name: "Ada" } },
        { id: "t2", title: "Build", assignee: { id: "u2", name: "Grace" } },
      ],
    },
    {
      id: "p2",
      name: "Gemini",
      tasks: [{ id: "t3", title: "Plan", assignee: { id: "u1", name: "Ada" } }],
    },
  ],
};

function l87Normalise(data) {
  const projects = {};
  const tasks = {};
  const users = {};

  for (const project of data.projects) {
    projects[project.id] = { id: project.id, name: project.name, taskIds: [] };
    for (const task of project.tasks) {
      tasks[task.id] = { id: task.id, title: task.title, assigneeId: task.assignee.id, projectId: project.id };
      projects[project.id].taskIds.push(task.id);
      users[task.assignee.id] = { id: task.assignee.id, name: task.assignee.name };
    }
  }

  return { projects, tasks, users };
}

const l87Flat = l87Normalise(l87Nested);
console.log("projects:", Object.keys(l87Flat.projects));
console.log("tasks   :", Object.keys(l87Flat.tasks));
console.log("users   :", l87Flat.users);

// --- the same edit, both ways: rename user u1 -------------------------------
console.log("\nrenaming Ada to 'Ada L.'");

// nested: every copy of the user has to be found and updated
const l87NestedRenamed = {
  projects: l87Nested.projects.map((project) => ({
    ...project,
    tasks: project.tasks.map((task) =>
      task.assignee.id === "u1" ? { ...task, assignee: { ...task.assignee, name: "Ada L." } } : task,
    ),
  })),
};
const l87CopiesTouched = l87Nested.projects.flatMap((p) => p.tasks).filter((t) => t.assignee.id === "u1").length;
console.log("  nested    — copies of the user that had to change:", l87CopiesTouched);

// normalised: one place
const l87FlatRenamed = { ...l87Flat, users: { ...l87Flat.users, u1: { ...l87Flat.users.u1, name: "Ada L." } } };
console.log("  normalised— copies of the user that had to change: 1");
console.log("  result:", l87FlatRenamed.users.u1);

// and the reason it matters beyond keystrokes:
console.log(
  "\n  nested: the same user exists",
  l87CopiesTouched,
  "times, so two of them CAN disagree. normalised: there is one, so they cannot.",
);
console.log("  nested rename applied correctly?", l87NestedRenamed.projects[0].tasks[0].assignee.name === "Ada L.");

### Exercise

Parts 1 and 2 are **runnable — plain JS**; part 3 is a written answer.

1. Write `l87Denormalise(flat, projectId)` that rebuilds one project in the nested shape, tasks
   and assignees included. This is what a selector does for a component that wants a whole
   project — and writing it shows you that normalising loses nothing.
2. Add a task to project `p1` in both shapes. Count how many objects you had to copy in each, and
   say which one you would rather write a reducer for.
3. For each of these, decide Context or a store, in one line each: a theme toggle · the logged-in
   user · a shopping cart used by six screens · the open/closed state of one dropdown · a list of
   projects with tasks, edited from three different screens · the current locale.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** Normalisation has a cost too, and pretending otherwise is how people end
up hating it.

Using the normalised shape, write the three lookups a UI actually needs: every task in a project
with its assignee's name, every task assigned to one user across all projects, and the number of
tasks per project.

Then answer in comments:

1. Which of the three was harder than it would have been with the nested shape, and why?
2. All three of your functions take the whole state and return derived data. What are they called
   in Redux vocabulary, and which earlier lesson is that idea (hint: it is not a new one)?
3. One of your lookups scans every task to find the ones for a user. What would you add to the
   state to make that fast — and what does adding it cost you the next time a task is reassigned?

In [ ]:
// Your code here

## LESSON 88 — `createSlice`

A slice is one part of the store — the tasks, the filters, the current user — with its state, its
reducers and its action creators defined together in one file.

**The React API:**

```js
import { createSlice } from "@reduxjs/toolkit";

const tasksSlice = createSlice({
  name: "tasks",
  initialState: { items: [], status: "idle", error: null },
  reducers: {
    added: (state, action) => {
      state.items.push({ id: action.payload.id, text: action.payload.text });
    },
    cleared: (state) => {
      state.items = [];
    },
  },
});

export const { added, cleared } = tasksSlice.actions;
export default tasksSlice.reducer;
```

State and reducers in; **action creators and a reducer out**. You write the `reducers` object;
Redux Toolkit derives everything else from it.

### What it gives you, measured

Running exactly that slice:

```text
actions exported:      [ 'added', 'cleared' ]
action creator output: { type: 'tasks/added', payload: { id: 1, text: 'write' } }
slice.reducer:         function
```

So `added({ id: 1, text: "write" })` builds `{ type: "tasks/added", payload: … }` — the action
object you wrote by hand in LESSON 53, now generated, with the type namespaced by the slice's
`name` so two slices cannot collide.

### The part that looks wrong

`state.items.push(...)` in a reducer. Everything since LESSON 27 has said never to mutate state,
and LESSON 52 said a reducer must be pure. Both are still true. Redux Toolkit's explanation:

> Redux Toolkit allows us to write "mutating" logic in reducers. It doesn't actually mutate the
> state because it uses the Immer library, which detects changes to a "draft state" and produces a
> brand new immutable state based off those changes.

The `state` your reducer receives is a **draft** — a proxy that records what you did to it. Immer
then produces a new object from that recording. Measured against the slice above:

```text
before.items.length: 0 | after.items.length: 1
new object?                      true
unknown action returns identical object: true
```

The state you were given is untouched, the result is a new object, and — the detail worth
noticing — an action the slice does not handle returns the **very same object**, so nothing that
compares by identity (`memo`, `useSelector`, a dependency array) sees a change.

Two rules follow, and breaking either is the classic Immer bug:

- **Either mutate the draft or return a new value — never both.** `state.items.push(x)` is fine;
  `return { ...state }` is fine; doing both in one reducer is not.
- **Only inside a reducer.** The draft exists for the duration of the call. Redux Toolkit freezes
  the real state in development, so mutating it elsewhere throws — measured:
  `TypeError: Cannot add property 1, object is not extensible`.

### It is still the reducer you already know

The clearest way to see `createSlice` is as generated code. This, written by hand, is what the
slice above amounts to — and it is the LESSON 52 shape exactly:

```js
function tasksReducer(state = initialState, action) {
  switch (action.type) {
    case "tasks/added":
      return { ...state, items: [...state.items, action.payload] };
    case "tasks/cleared":
      return { ...state, items: [] };
    default:
      return state;
  }
}
const added = (payload) => ({ type: "tasks/added", payload });
```

`createSlice` removes the action-type strings, the switch, the spreads and the `default` case.
It does not remove the concept: **a pure function from state and action to the next state.**

### Key Notes

- `createSlice({ name, initialState, reducers })` → `slice.actions` and `slice.reducer`.
- Action types are namespaced automatically: `tasks/added`.
- The `state` in a reducer is an Immer draft. Mutate it **or** return a new value, never both, and
  only inside the reducer.
- An unhandled action returns the identical state object, so identity comparisons stay meaningful.

### Example

**Runnable — plain JS.** The hand-written equivalent, because that is the part that transfers and
the part a notebook can honestly run. Write this and `createSlice` stops being magic.

In [ ]:
// L88 — the reducer createSlice would generate

const l88Initial = { items: [], status: "idle", error: null };

function l88Reducer(state = l88Initial, action) {
  switch (action.type) {
    case "tasks/added":
      return { ...state, items: [...state.items, action.payload] };
    case "tasks/toggled":
      return {
        ...state,
        items: state.items.map((item) =>
          item.id === action.payload ? { ...item, done: !item.done } : item,
        ),
      };
    case "tasks/cleared":
      return { ...state, items: [] };
    default:
      return state;                       // the same object, deliberately
  }
}

// the action creators createSlice would export
const l88Added = (task) => ({ type: "tasks/added", payload: task });
const l88Toggled = (id) => ({ type: "tasks/toggled", payload: id });

console.log("action object:", l88Added({ id: "t1", text: "write", done: false }));

const l88One = l88Reducer(l88Initial, l88Added({ id: "t1", text: "write", done: false }));
const l88Two = l88Reducer(l88One, l88Added({ id: "t2", text: "review", done: false }));
console.log("after two adds:", l88Two.items.map((t) => t.id));

const l88Toggle = l88Reducer(l88Two, l88Toggled("t1"));
console.log("t1 done?", l88Toggle.items.find((t) => t.id === "t1").done);

// the three properties that make it a reducer
console.log("\ninput untouched:      ", l88Initial.items.length === 0);
console.log("new object returned:  ", l88Toggle !== l88Two);
console.log("unhandled action -> identical object:", l88Reducer(l88Toggle, { type: "other/thing" }) === l88Toggle);

// and the property that makes identity comparisons work: the items array of an UNCHANGED
// part of the state keeps its reference
const l88StatusChange = l88Reducer(l88Toggle, { type: "other/thing" });
console.log("items array reused when nothing changed:", l88StatusChange.items === l88Toggle.items);

### Exercise

Parts 1 and 2 are **runnable — plain JS**; part 3 is **in your project**.

1. Add `tasks/removed` and `tasks/renamed` to `l88Reducer`, keeping every rule above true. Then
   write three assertions that prove it: the input is untouched, a new object comes back, and an
   unhandled action returns the identical object.
2. Write the same reducer a second time as `l88Draft(state, action)` where you are *allowed* to
   mutate — the Immer style — and then a tiny `l88Apply(state, action)` that copies the state
   deeply, runs `l88Draft` on the copy and returns it. You have now written the outline of what
   Immer does for you, and you can compare the two reducers side by side. Which reads better for
   the `renamed` case?
3. **In your project:** install `@reduxjs/toolkit` and `react-redux` in a scratch project, write
   the slice from this lesson, and log `slice.actions`, an action creator's output, and the result
   of dispatching an unhandled action to `slice.reducer`. Check the three numbers against the
   measured output above.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** Action naming, which is the part of Redux that people get wrong for years.

LESSON 54 said actions are **events, named in the past tense** — things that happened, not
commands. Here are ten action types from a real-ish codebase. Classify each as an event or a
command, rewrite the commands, and say what changes about the reducer:

```
tasks/setItems        tasks/added           ui/openModal
tasks/fetchSucceeded  filters/setFilter     auth/loggedOut
tasks/updateTaskText  projects/selected     ui/toggleSidebar
tasks/removeAllDone
```

Then answer in comments:

1. `tasks/setItems` is the most common of these in real code, and the most damaging. What does a
   reducer that handles it actually know, and what can a DevTools log of ten `setItems` actions
   tell you?
2. Two of the ten are fine as commands even under a strict reading. Which, and what makes UI
   actions different?

In [ ]:
// Your code here

## LESSON 89 — The store, the Provider, and selectors

You have a slice. This lesson connects it to React — three small pieces of setup and two Hooks —
and then spends most of its time on selectors, which are where the thinking actually is.

### The store

**The React API:**

```js
// src/store.js
import { configureStore } from "@reduxjs/toolkit";
import tasksReducer from "./features/tasks/tasksSlice.js";
import filtersReducer from "./features/filters/filtersSlice.js";

export const store = configureStore({
  reducer: {
    tasks: tasksReducer,
    filters: filtersReducer,
  },
});
```

The `reducer` object *is* the shape of your state: this store's state is
`{ tasks: …, filters: … }`, each key handled by its own slice. Redux Toolkit's own description is
that `configureStore` **"automatically sets up the store with good default settings"** — including
the Redux DevTools connection and, in development, the checks that catch mutations and
non-serialisable values.

### The Provider

```jsx
// src/main.jsx
import { Provider } from "react-redux";
import { store } from "./store.js";

root.render(
  <Provider store={store}>
    <App />
  </Provider>,
);
```

One provider, at the root, for the whole store — unlike Context, where LESSON 57 told you to have
one per concern. The store's own structure does the separating.

### Reading and dispatching

```jsx
import { useDispatch, useSelector } from "react-redux";
import { added } from "./features/tasks/tasksSlice.js";

function TaskList() {
  const tasks = useSelector((state) => state.tasks.items);
  const dispatch = useDispatch();

  return (
    <>
      <ul>{tasks.map((task) => <li key={task.id}>{task.text}</li>)}</ul>
      <button onClick={() => dispatch(added({ id: crypto.randomUUID(), text: "New task" }))}>
        add
      </button>
    </>
  );
}
```

`useSelector` runs your function against the store's state and **subscribes the component to what
it returned**. `useDispatch` gives you `dispatch`, and you pass it the action creator's result.

### Selectors, and the one rule

A selector is a pure function from state to something a component needs. That is all — and you
have been writing them since LESSON 21 and 29 under the name "derived values".

Two reasons to name them rather than inline them:

```js
// src/features/tasks/selectors.js
export const selectAllTasks = (state) => state.tasks.items;
export const selectTaskCount = (state) => state.tasks.items.length;
export const selectVisibleTasks = (state) =>
  state.tasks.items.filter((task) => (state.filters.done === "all" ? true : task.done));
```

First, the component stops knowing the shape of the store. Move `items` under
`state.tasks.list.items` and one file changes instead of nine. Second, they are pure functions, so
they are the cheapest thing in the app to test — LESSON 85 exactly.

**The rule that costs people an afternoon:** `useSelector` compares the returned value with
`Object.is` (LESSON 40 and 69, again). Return a **new object or array every time and the component
re-renders on every dispatch in the app**, whether or not anything it cares about changed:

```js
// 🔴 a new array on every call — re-renders on every action, anywhere
const visible = useSelector((state) => state.tasks.items.filter((t) => !t.done));

// ✅ select the raw value, derive after
const items = useSelector((state) => state.tasks.items);
const visible = items.filter((task) => !task.done);
```

The general form of the fix: **select the smallest raw thing, then derive**. When the derivation is
genuinely expensive, `createSelector` from Redux Toolkit memoizes it — the same
make-it-correct-then-measure order as topic 23, and not something to reach for by default.

### Key Notes

- `configureStore({ reducer: { … } })` — the keys are the state's shape; DevTools come free.
- One `<Provider store>` at the root; the store's structure replaces one-provider-per-concern.
- `useSelector(fn)` subscribes to what `fn` returns; `useDispatch()` gives you `dispatch`.
- A selector returning a **new** array or object on every call re-renders the component on every
  action. Select raw, derive after.

### Example

**Runnable — plain JS.** Selectors are pure functions of a plain object, so all of this runs here
— including the identity trap, which you can see without React at all.

In [ ]:
// L89 — selectors, and the identity trap

const l89State = {
  tasks: {
    items: [
      { id: "t1", text: "Design", done: true, projectId: "p1" },
      { id: "t2", text: "Build", done: false, projectId: "p1" },
      { id: "t3", text: "Plan", done: false, projectId: "p2" },
    ],
    status: "success",
  },
  filters: { done: "open", projectId: "p1" },
};

// plain selectors
const l89SelectAll = (state) => state.tasks.items;
const l89SelectStatus = (state) => state.tasks.status;
const l89SelectCount = (state) => state.tasks.items.length;

// a selector that derives
const l89SelectVisible = (state) =>
  state.tasks.items.filter(
    (task) =>
      (state.filters.done === "all" || (state.filters.done === "open") === !task.done) &&
      task.projectId === state.filters.projectId,
  );

console.log("all      :", l89SelectAll(l89State).map((t) => t.id));
console.log("count    :", l89SelectCount(l89State));
console.log("visible  :", l89SelectVisible(l89State).map((t) => t.id));

// --- the identity trap ------------------------------------------------------
// the store's state has NOT changed; we call the same selector twice
const l89A = l89SelectVisible(l89State);
const l89B = l89SelectVisible(l89State);

console.log("\nsame contents?", JSON.stringify(l89A) === JSON.stringify(l89B));
console.log("same reference?", l89A === l89B, "  <- useSelector compares THIS");
console.log("so the component would re-render even though nothing changed.");

// selecting a raw value is stable
console.log("\nraw selector, twice:", l89SelectAll(l89State) === l89SelectAll(l89State));
console.log("primitive selector, twice:", l89SelectCount(l89State) === l89SelectCount(l89State));

// --- what memoization would do ----------------------------------------------
function l89Memoize(selector) {
  let lastState = null;
  let lastResult;
  return (state) => {
    if (state !== lastState) {
      lastState = state;
      lastResult = selector(state);
    }
    return lastResult;
  };
}

const l89SelectVisibleMemo = l89Memoize(l89SelectVisible);
console.log(
  "\nmemoized selector, same state twice:",
  l89SelectVisibleMemo(l89State) === l89SelectVisibleMemo(l89State),
);

// this is the idea behind createSelector — and note it only helps because the store replaces the
// whole state object on every change, so `state !== lastState` is a reliable question to ask.

### Exercise

Parts 1 and 2 are **runnable — plain JS**; part 3 is **in your project**.

1. Write `l89SelectByProject(state, projectId)` and `l89SelectCountsByProject(state)` returning
   `{ p1: 2, p2: 1 }`. Then say, in a comment, which of the two is safe to use directly in
   `useSelector` and which is not, and why.
2. Your memoized selector above compares the whole state object. Break it: write a selector that
   depends only on `state.filters` and show that it recomputes when an unrelated part of the state
   changes. Then improve `l89Memoize` to take a list of input selectors and only recompute when one
   of *those* results changes — you have now written `createSelector`.
3. **In your project:** set up `configureStore`, the `<Provider>` and one component using
   `useSelector` and `useDispatch`. Then deliberately write the bad selector
   (`state => state.tasks.items.filter(...)` inline), add a console.log to the component, and
   dispatch an action from a *different* slice. Count the renders, then fix it and count again.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** Where should the logic live?

For each of these, decide whether it belongs in a **reducer**, a **selector**, or a **component**,
and write one line of justification:

```
1. adding a task to the list
2. showing only tasks for the selected project
3. sorting tasks by due date for display
4. marking every task in a project as done
5. formatting a date as "3 days ago"
6. counting how many tasks are still open
7. deciding whether the "clear completed" button is disabled
8. clamping a task's title to 80 characters when it is saved
```

Then answer in comments: two of these could reasonably go in either a reducer or a selector, and
the choice has a consequence. Name them, and state the rule you would use — the question to ask is
*"is this a fact about the data, or a decision about this screen?"*

In [ ]:
// Your code here

## LESSON 90 — `createAsyncThunk`

Everything in a reducer is synchronous — LESSON 52's rule, and Redux's. So where does a fetch go?

Into a **thunk**: a function you dispatch, which does the async work and dispatches ordinary
actions as it goes. Redux Toolkit generates one for you.

**The React API:**

```js
import { createAsyncThunk, createSlice } from "@reduxjs/toolkit";
import { getTasks } from "../../services/tasks.js";

export const fetchTasks = createAsyncThunk("tasks/fetch", async (projectId, { rejectWithValue }) => {
  try {
    return await getTasks(projectId);          // becomes action.payload on success
  } catch (error) {
    return rejectWithValue(error.message);     // becomes action.payload on failure
  }
});
```

The docs describe it as a function that

> generates promise lifecycle action types based on the action type prefix that you pass in, and
> returns a thunk action creator that will run the promise callback and dispatch the lifecycle
> actions based on the returned promise.

Three action types, from the one prefix. Measured:

```text
tasks/fetch/pending | tasks/fetch/fulfilled | tasks/fetch/rejected
```

### Handling them: `extraReducers`

They are not in your `reducers` object, because they were not defined there — so they arrive
through `extraReducers`, which is where a slice responds to actions defined elsewhere:

```js
const tasksSlice = createSlice({
  name: "tasks",
  initialState: { items: [], status: "idle", error: null },
  reducers: {},
  extraReducers: (builder) => {
    builder
      .addCase(fetchTasks.pending, (state) => {
        state.status = "loading";
        state.error = null;
      })
      .addCase(fetchTasks.fulfilled, (state, action) => {
        state.status = action.payload.length === 0 ? "empty" : "success";
        state.items = action.payload;
      })
      .addCase(fetchTasks.rejected, (state, action) => {
        state.status = "error";
        state.error = action.payload ?? action.error.message;
      });
  },
});
```

Look at what that is: **LESSON 46's four states**, written as three cases. Redux does not change
the model you already have — it moves it out of the component.

Measured, dispatching the thunk and recording `status` after every store notification:

| path | statuses seen | final state |
|---|---|---|
| success | `["loading", "success"]` | `items: [{ id: 9, … }], error: null` |
| failure | `["loading", "error"]` | `items: [], error: "server said no"` |

### `rejectWithValue`, and why `action.payload ?? action.error.message`

If the thunk **throws**, the rejected action carries `action.error` — a serialised version of the
error, whose `message` is often something like `"Request failed with status code 500"`.

If the thunk returns `rejectWithValue(x)`, then `x` is `action.payload`, and it is yours: a status
code, a field-level error object, a message you wrote for the user (LESSON 76). That is why the
reducer above prefers the payload and falls back to the error message.

### Dispatching it

```jsx
const dispatch = useDispatch();
const status = useSelector((state) => state.tasks.status);

useEffect(() => {
  if (status === "idle") {
    dispatch(fetchTasks(projectId));
  }
}, [status, projectId, dispatch]);
```

Still an Effect, still LESSON 47 — synchronising with something outside React. What changed is
where the *result* goes.

`dispatch(thunk())` returns a Promise, so a component can `await` it when it needs to do something
afterwards. It resolves for both outcomes; `.unwrap()` makes it throw on rejection if you would
rather use `try`/`catch`.

### One pointer, then stop

Redux Toolkit also ships **RTK Query**, which generates the whole loading/error/caching layer from
an endpoint definition and removes most of what this lesson just wrote by hand. It is the right
tool for an app whose main job is server data — and it is out of scope here, because it is a
substantial API of its own and Mini-project 4 needs exactly one thunk. Now that you have written
the thing it replaces, you will be able to read its documentation in an afternoon.

### Key Notes

- Async work goes in a thunk; reducers stay synchronous.
- `createAsyncThunk(prefix, payloadCreator)` generates `pending`, `fulfilled` and `rejected`.
- Handle them in `extraReducers` with `builder.addCase` — the same four states as LESSON 46.
- `rejectWithValue(x)` puts your own value on `action.payload`; otherwise you get
  `action.error.message`.

### Example

**Runnable — plain JS.** The reducer half is pure and transferable, so it runs here: a
`(state, action) => state` that handles three lifecycle actions by name, driven by a mock async
call. This is exactly what `extraReducers` does, minus the builder syntax.

In [ ]:
// L90 — the four states, driven by lifecycle actions

const l90Initial = { items: [], status: "idle", error: null };

function l90Reducer(state = l90Initial, action) {
  switch (action.type) {
    case "tasks/fetch/pending":
      return { ...state, status: "loading", error: null };
    case "tasks/fetch/fulfilled":
      return {
        ...state,
        status: action.payload.length === 0 ? "empty" : "success",
        items: action.payload,
      };
    case "tasks/fetch/rejected":
      return { ...state, status: "error", error: action.payload ?? action.error.message };
    default:
      return state;
  }
}

// what createAsyncThunk does, in the small: run the promise, dispatch three kinds of action
async function l90RunThunk(payloadCreator, arg, dispatch) {
  dispatch({ type: "tasks/fetch/pending" });
  try {
    const payload = await payloadCreator(arg, {
      rejectWithValue: (value) => ({ __rejected: true, value }),
    });
    if (payload && payload.__rejected) {
      dispatch({ type: "tasks/fetch/rejected", payload: payload.value });
    } else {
      dispatch({ type: "tasks/fetch/fulfilled", payload });
    }
  } catch (error) {
    dispatch({ type: "tasks/fetch/rejected", error: { message: error.message } });
  }
}

// a tiny dispatcher that records the state after every action
function l90Track() {
  let state = l90Initial;
  const seen = [];
  return {
    dispatch: (action) => {
      state = l90Reducer(state, action);
      seen.push(state.status);
    },
    result: () => ({ statuses: seen, state }),
  };
}

const l90Mock = (mode) => async (_arg, { rejectWithValue }) => {
  await new Promise((resolve) => setTimeout(resolve, 10));
  if (mode === "fail") return rejectWithValue("server said no");
  if (mode === "throw") throw new Error("Request failed with status code 500");
  if (mode === "none") return [];
  return [{ id: "t9", text: "from the server" }];
};

for (const mode of ["ok", "none", "fail", "throw"]) {
  const tracker = l90Track();
  await l90RunThunk(l90Mock(mode), "p1", tracker.dispatch);
  const { statuses, state } = tracker.result();
  console.log(
    `${mode.padEnd(6)} statuses: ${JSON.stringify(statuses).padEnd(24)} items: ${state.items.length}  error: ${JSON.stringify(state.error)}`,
  );
}

### Exercise

Parts 1 and 2 are **runnable — plain JS**; part 3 is **in your project**.

1. Add a second thunk's actions — `tasks/save/pending|fulfilled|rejected` — to `l90Reducer`,
   keeping `status` for the *load* and adding a separate `saving` flag. Then explain in a comment
   why one shared `status` field for both would be a bug, using LESSON 67's argument about states
   that can contradict.
2. The reducer sets `status: "empty"` for an empty array. Write the four assertions that pin all
   four states down, and then answer: which of the four does a component that only checks
   `if (loading) … else render(items)` get wrong, and what does the user see?
3. **In your project:** write the real slice with `createAsyncThunk` and `extraReducers`, dispatch
   it, and log the statuses you see. Compare them with the measured table in this lesson. Then
   make the thunk fail with `rejectWithValue("server said no")` and check that `action.payload`
   carries it.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** Two requests, one store — the race from LESSON 43, in a new place.

The user clicks project `p1`, then quickly clicks `p2`. Two thunks are in flight and `p1`'s
response arrives **last**.

1. Show the bug with the reducer above: simulate both, and print the final state. The store now
   shows `p2` selected and `p1`'s tasks.
2. Fix it. Add a `requestId` to each lifecycle action, keep the current one in the state, and
   ignore `fulfilled`/`rejected` actions whose id is not current. Prove it works with the same
   sequence. (Real `createAsyncThunk` puts a `requestId` in `action.meta` for exactly this.)
3. In a comment: `AbortController` (LESSON 43) would also solve this, and `createAsyncThunk` gives
   the payload creator a `signal`. Which of the two approaches — ignore the stale answer, or abort
   the stale request — would you use here, and what does each one cost?

In [ ]:
// Your code here

> **Topic 27 complete — LESSON 87 to 90.** When a store earns its place, `createSlice`, the store
> and selectors, and one async thunk with the four states you have used since topic 15.
>
> Two lessons left: production builds and deployment. Then Mini-project 4, which uses this store.